In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO

def finviz_screener_scraper(filters: dict, page: int = 1):
    """
    Realiza web scraping do Finviz sem depender de arquivo de settings.
    """
    base_url = "https://finviz.com/screener.ashx?"
    
    # Parâmetros base do Finviz: v=111 (visão geral), ft=4 (filtros avançados)
    params = {'v': '111', 'ft': '4', 'r': (page-1)*20+1} 
    
    # 1. Mapeamento de filtros (Embutido para facilitar)
    # Se quiser adicionar mais, siga o padrão: 'Nome Amigável': 'prefixo_no_finviz'
    FINVIZ_MAP = {
        "IPO Date": "ipodate_",
        "Country": "ctry_",
        "Sector": "sec_",
        "Industry": "ind_",
        "Index": "idx_"
    }

    filter_list = []
    for key, value in filters.items():
        # Se a chave estiver no mapa, usa o prefixo, senão usa a chave como está
        prefix = FINVIZ_MAP.get(key, key.lower().replace(' ', '') + "_")
        
        # Formata o código do filtro (ex: ipodate_more25)
        clean_value = str(value).lower().replace(' ', '').replace('-', '')
        filter_list.append(f"{prefix}{clean_value}")

    if filter_list:
        params['f'] = ",".join(filter_list)
    
    # 2. Construir URL e Fazer Request
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # O Finviz coloca os resultados em um <tr> com id 'screener-table'
        screener_tr = soup.find('tr', id='screener-table')
        if not screener_tr:
            return pd.DataFrame()
            
        table_container = screener_tr.find('table', class_='styled-table-new')
        if not table_container:
            return pd.DataFrame()
        
        # 3. Processar a Tabela
        df_list = pd.read_html(StringIO(str(table_container)))
        if not df_list:
            return pd.DataFrame()
            
        result_df = df_list[0]
        
        # Renomear colunas para o padrão
        result_df.columns = ["No.", "Ticker", "Company", "Sector", "Industry", "Country", 
                             "Market Cap", "P/E", "Price", "Change", "Volume"]
        
        return result_df

    except Exception as e:
        print(f"Erro ao processar página {page}: {e}")
        return pd.DataFrame()

def finviz_screener_scraper_all_pages(filters: dict):
    page = 1
    all_results = []
    
    while True:
        print(f"Coletando página {page}...")
        df = finviz_screener_scraper(filters, page=page)
        
        if df.empty:
            break
            
        all_results.append(df)
        page += 1
        
        # O Finviz costuma limitar o número de resultados. 
        # Se a página vier com menos de 20 linhas, é a última.
        if len(df) < 20:
            break
            
    if not all_results:
        return pd.DataFrame()
        
    return pd.concat(all_results).reset_index(drop=True)

In [4]:
# A chave "IPO Date" vai virar "ipodate_" e o valor "more25" vai completar
filtros = {
    "IPO Date": "more5",
    "Country": "usa"
}

df_finance = finviz_screener_scraper_all_pages(filtros)
print(df_finance.head())

Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Coletando página 15...
Coletando página 16...
Coletando página 17...
Coletando página 18...
Coletando página 19...
Coletando página 20...
Coletando página 21...
Coletando página 22...
Coletando página 23...
Coletando página 24...
Coletando página 25...
Coletando página 26...
Coletando página 27...
Coletando página 28...
Coletando página 29...
Coletando página 30...
Coletando página 31...
Coletando página 32...
Coletando página 33...
Coletando página 34...
Coletando página 35...
Coletando página 36...
Coletando página 37...
Coletando página 38...
Coletando página 39...
Coletando página 40...
Coletando página 41...
Coletando página 42...
Coletando página 43...
Coletando página 44.

In [7]:
df_finance[df_finance["Country"]=="USA"].to_csv("metadata_att.csv")

## Filtering by IPO Date

In [22]:
df = pd.read_csv("metadata_att - metadata_att.csv.csv")
df

,Ticker,Company,Sector,Industry,Country,ipo_date,mcap_2015
0,A,Agilent Technologies Inc,Healthcare,Diagnostics & Research,USA,18/11/1999,NaN
1,AA,Alcoa Corp,Basic Materials,Aluminum,USA,01/11/2016,NaN
2,AAA,Alternative Access First Priority CLO Bond ETF,Financial,Exchange Traded Fund,USA,09/09/2020,NaN
3,AAAU,Goldman Sachs Physical Gold ETF,Financial,Exchange Traded Fund,USA,15/08/2018,NaN
4,AADR,AdvisorShares Dorsey Wright ADR ETF,Financial,Exchange Traded Fund,USA,21/07/2010,NaN
...,...,...,...,...,...,...,...
5655,ZTS,Zoetis Inc,Healthcare,Drug Manufacturers - Specialty & Generic,USA,01/02/2013,NaN
5656,ZUMZ,Zumiez Inc,Consumer Cyclical,Apparel Retail,USA,06/05/2005,NaN
5657,ZVRA,Zevra Therapeutics Inc,Healthcare,Biotechnology,USA,16/04/2015,NaN
5658,ZWS,Zurn Elkay Water Solutions Corp,Industrials,Pollution & Treatment Controls,USA,29/03/2012,NaN


In [24]:
# 1. Converter a coluna 'ipo_date' para o formato datetime
# O format='%d/%m/%Y' reflete o formato dia/mês/ano que está no seu dataframe.
# errors='coerce' transforma eventuais datas mal formatadas em NaT (not a time)
df['ipo_date'] = pd.to_datetime(df['ipo_date'], format='%d/%m/%Y', errors='coerce')

# 2. Criar a coluna 'ipo_year' extraindo apenas o ano
df['ipo_year'] = df['ipo_date'].dt.year
df = df.dropna(subset=['ipo_year'])
df['ipo_year'] = df['ipo_year'].astype(int)

# 3. Filtrar o DataFrame para manter apenas as empresas com IPO até 2019
df_filtrado = df[df['ipo_year'] <= 2019].reset_index()

# Exibir o resultado
df_filtrado.head()

/tmp/ipykernel_27320/49705491.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ipo_year'] = df['ipo_year'].astype(int)


,index,Ticker,Company,Sector,Industry,Country,ipo_date,mcap_2015,ipo_year
0,0,A,Agilent Technologies Inc,Healthcare,Diagnostics & Research,USA,1999-11-18,NaN,1999
1,1,AA,Alcoa Corp,Basic Materials,Aluminum,USA,2016-11-01,NaN,2016
2,3,AAAU,Goldman Sachs Physical Gold ETF,Financial,Exchange Traded Fund,USA,2018-08-15,NaN,2018
3,4,AADR,AdvisorShares Dorsey Wright ADR ETF,Financial,Exchange Traded Fund,USA,2010-07-21,NaN,2010
4,5,AAL,American Airlines Group Inc,Industrials,Airlines,USA,2005-09-27,NaN,2005


In [26]:
df_filtrado.to_csv("metadata_att.csv")

## Returns

In [1]:
import pandas as pd 

returns = pd.read_parquet("../../data/01_raw/returns.parquet")

## Salvando por ano

In [7]:
import pandas as pd 
import numpy as np
from datetime import date

returns = pd.read_parquet("../../data/01_raw/returns.parquet")

k = 5
year = 2014
returns_year = returns.loc[f"{year-k+1}-01-01":f"{year}-12-31"]
returns_year.loc[:, ~returns_year.isna().any()]

# Dividing by years
for year in range(2014,2026):
    returns_year = returns.loc[f"{year-k+1}-01-01":f"{year}-12-31"]
    returns_year = returns_year.loc[:, ~returns_year.isna().any()]
    cols_year = returns_year.columns
    returns_year.to_parquet(f"../../data/02_clean/returns_new_{year-k+1}_{year}.parquet")

    # Only 1 year
    returns_year = returns.loc[f"{year}-01-01":f"{year}-12-31"]
    returns_year = returns_year.loc[:, cols_year]
    returns_year.to_parquet(f"../../data/02_clean/returns_new_{year}.parquet")

In [4]:
import pandas as pd 
import numpy as np
from datetime import date

# Dividing by years
for year in range(2014,2026):
    returns_year = returns.loc[f"{year}-01-01":f"{year}-12-31"]
    returns_year = returns_year.loc[:, ~returns_year.isna().any()]
    returns_year.to_parquet(f"../../data/02_clean/returns_new_{year}.parquet")